# CV Auto-DL Codegen Demo

This notebook runs the CIFAR-10 Hugging Face input through candidate selection, baseline generation, ablation, targeted refinement, review, notebook export, and a small real Colab training run.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

REPO_URL = os.getenv('JIAOZI_REPO_URL', 'https://github.com/Isso-W/Jiaozi.git')
REPO_DIR = Path('/content/Jiaozi')
repo_root = Path.cwd()

if not (repo_root / 'cv_autodl_agent').exists():
    if not (REPO_DIR / 'cv_autodl_agent').exists():
        if REPO_DIR.exists():
            subprocess.run(['rm', '-rf', str(REPO_DIR)], check=True)
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
    repo_root = REPO_DIR

if not (repo_root / 'cv_autodl_agent').exists():
    raise RuntimeError(f'Could not find cv_autodl_agent in {repo_root}. Push the latest local project to GitHub or upload the full repo to Colab.')

os.chdir(repo_root)
sys.path.insert(0, str(repo_root))
print('Repository root:', repo_root)

In [ ]:
from cv_autodl_agent import CVAutoDLWorkflow, DatasetManifest, RetrievedModelCandidate
from cv_autodl_agent.io_utils import read_json

manifest = DatasetManifest.from_dict(read_json('examples/cifar10_manifest.json'))
candidates = [RetrievedModelCandidate.from_dict(item) for item in read_json('examples/cifar10_candidates.json')]

print(json.dumps({
    'dataset_manifest': manifest.to_dict(),
    'candidate_model_ids': [candidate.model_id for candidate in candidates]
}, indent=2))

In [ ]:
workflow = CVAutoDLWorkflow()
result = workflow.run(
    manifest=manifest,
    candidates=candidates,
    output_root='colab_demo_output',
    execution_mode='simulate',
    notebook_execution_mode='real',
)

summary = {
    'selected_model': result.selected_candidate.model_id,
    'baseline': {
        'metric': result.baseline_result.primary_metric_name,
        'value': result.baseline_result.primary_metric_value,
    },
    'final': {
        'metric': result.final_result.primary_metric_name,
        'value': result.final_result.primary_metric_value,
    },
    'ablation_best_component': result.ablation_summary.best_component_to_change,
    'recommended_edit_region': result.ablation_summary.recommended_edit_region,
    'review_status': result.review_report.status,
    'generated_notebook': result.notebook_path,
}
print(json.dumps(summary, indent=2))

In [ ]:
import pandas as pd

ablation_rows = result.ablation_summary.tested_variants
pd.DataFrame(ablation_rows)[['component', 'label', 'metric_value', 'delta_vs_baseline']]

In [ ]:
project_dir = Path(result.project_dir)
print('Generated project:', project_dir)
for path in sorted(project_dir.iterdir()):
    print(path.relative_to(project_dir))

In [ ]:
!pip install -q -r {project_dir}/requirements.txt
!python {project_dir}/train.py --manifest {project_dir}/manifest.json --spec {project_dir}/refined_training_spec.json --output-dir colab_demo_output/real_cifar10_train --execution-mode real

In [ ]:
metrics_path = Path('colab_demo_output/real_cifar10_train/metrics.json')
checkpoint_path = Path('colab_demo_output/real_cifar10_train/checkpoints/best.pt')
print(json.dumps(json.loads(metrics_path.read_text()), indent=2))
print('checkpoint_exists:', checkpoint_path.exists())